## Category tree (2 tiers)

`categories.csv` is a **3-level** tree: **Apparel / Shoes / Accessories** (roots) → intermediate categories (e.g. Tops, Bags) → leaf groups (e.g. T-Shirts, Backpacks).

This notebook **drops the root tier** and lists each **intermediate category** with its **direct children** only.

In [1]:
from pathlib import Path

import pandas as pd

CATEGORIES_CSV = Path("test_items_set") / "categories.csv"

ROOT_IDS = {
    "00000000-0000-0000-0000-000000000001",  # Apparel
    "00000000-0000-0000-0000-000000000002",  # Shoes
    "00000000-0000-0000-0000-000000000003",  # Accessories
}

df = pd.read_csv(CATEGORIES_CSV, dtype=str, keep_default_na=False)
df["parent_id"] = df["parent_id"].replace({"NULL": pd.NA, "": pd.NA})

# Tier 2: direct children of Apparel / Shoes / Accessories (omit those roots from the listing)
tier2 = df[df["parent_id"].isin(ROOT_IDS)].copy()
tier2["_sort"] = pd.to_numeric(tier2["sort_order"], errors="coerce")

dept_order = {
    "00000000-0000-0000-0000-000000000001": 0,
    "00000000-0000-0000-0000-000000000002": 1,
    "00000000-0000-0000-0000-000000000003": 2,
}
tier2["_dept"] = tier2["parent_id"].map(dept_order)
tier2 = tier2.sort_values(["_dept", "_sort", "name"], na_position="last")

rows = []
for _, parent in tier2.iterrows():
    pid = parent["id"]
    kids = df[df["parent_id"] == pid].copy()
    kids["_sort"] = pd.to_numeric(kids["sort_order"], errors="coerce")
    kids = kids.sort_values(["_sort", "name"], na_position="last")

    rows.append(
        {
            "category_id": pid,
            "category_name": parent["name"],
            "category_slug": parent["slug"],
            "child_count": len(kids),
            "children_names": "; ".join(kids["name"].tolist()),
            "children_slugs": "; ".join(kids["slug"].tolist()),
        }
    )

category_children_df = pd.DataFrame(rows)

# Optional: save a flat CSV for reuse
out_csv = Path("test_items_set") / "categories_two_tier_summary.csv"
category_children_df.to_csv(out_csv, index=False)
print(f"Wrote {out_csv.resolve()}\n")

for _, r in category_children_df.iterrows():
    print(f"{r['category_name']} ({r['category_slug']}) — {r['child_count']} children")
    if r["children_names"]:
        for name in r["children_names"].split("; "):
            print(f"  · {name}")
    else:
        print("  · (none)")
    print()

category_children_df

Wrote /Users/akudenko/Documents/FitFolio/fitfolio/fitfolio-ml-api/notebooks/test_items_set/categories_two_tier_summary.csv

Tops (tops) — 10 children
  · T-Shirts
  · Long Sleeve Tops
  · Shirts (Button-Up)
  · Polos
  · Tank Tops
  · Crop Tops
  · Bodysuits
  · Sweaters
  · Hoodies
  · Sweatshirts

Bottoms (bottoms) — 8 children
  · Jeans
  · Pants / Trousers
  · Chinos
  · Joggers
  · Sweatpants
  · Leggings
  · Shorts
  · Skirts

Dresses & Jumpsuits (dresses-and-jumpsuits) — 3 children
  · Dresses
  · Jumpsuits
  · Rompers

Outerwear (outerwear) — 6 children
  · Jackets
  · Coats
  · Parkas
  · Vests
  · Blazers
  · Windbreakers

Suits & Sets (suits-and-sets) — 3 children
  · Suits
  · Suit Separates (Blazers, Dress Pants)
  · Matching Sets

Activewear (activewear) — 5 children
  · Sports Bras
  · Workout Tops
  · Workout Bottoms
  · Compression
  · Tracksuits

Underwear & Sleepwear (underwear-and-sleepwear) — 5 children
  · Underwear
  · Bras
  · Socks
  · Sleepwear / Pajamas
  · L

,category_id,category_name,category_slug,child_count,children_names,children_slugs
0,00000000-0000-0000-0000-000000000101,Tops,tops,10,T-Shirts; Long Sleeve Tops; Shirts (Button-Up)...,t-shirts; long-sleeve-tops; button-up-shirts; ...
1,00000000-0000-0000-0000-000000000102,Bottoms,bottoms,8,Jeans; Pants / Trousers; Chinos; Joggers; Swea...,jeans; pants_and_trousers; chinos; joggers; sw...
2,00000000-0000-0000-0000-000000000103,Dresses & Jumpsuits,dresses-and-jumpsuits,3,Dresses; Jumpsuits; Rompers,dresses; jumpsuits; rompers
3,00000000-0000-0000-0000-000000000104,Outerwear,outerwear,6,Jackets; Coats; Parkas; Vests; Blazers; Windbr...,jackets; coats; parkas; vests; blazers; windbr...
4,00000000-0000-0000-0000-000000000105,Suits & Sets,suits-and-sets,3,"Suits; Suit Separates (Blazers, Dress Pants); ...",suits; suit-separates; matching-sets
5,00000000-0000-0000-0000-000000000106,Activewear,activewear,5,Sports Bras; Workout Tops; Workout Bottoms; Co...,sports-bras; workout-tops; workout-bottoms; co...
6,00000000-0000-0000-0000-000000000107,Underwear & Sleepwear,underwear-and-sleepwear,5,Underwear; Bras; Socks; Sleepwear / Pajamas; L...,underwear; bras; socks; sleepwear-and-pajamas;...
7,00000000-0000-0000-0000-000000000108,Swimwear,swimwear,4,One-Piece; Bikini; Swim Trunks; Rash Guards,one-piece; bikini; swim-trunks; rash-guards
8,00000000-0000-0000-0000-000000000401,Athletic,athletic-shoes,4,Running Shoes; Training Shoes; Basketball Shoe...,running-shoes; training-shoes; basketball-shoe...
9,00000000-0000-0000-0000-000000000402,Casual,casual-shoes,2,Sneakers; Slip-ons,sneakers; slip-ons


## OpenAI metadata enrichment (vision)

Reads `test_items_set/test_items_predictions.csv`, sends each **image** plus coarse **category / gender / color**, and asks the model for tags, `itemName`, `description`, and **subCategory_1** / **subCategory_2** from the FitFolio 2-tier taxonomy (scoped to Apparel, Shoes, or Accessories).

Requires `OPENAI_API_KEY` (e.g. in `.env` at repo root or `fitfolio-ml-api`). Outputs `test_items_enriched.csv` and `test_items_enriched.jsonl`.

In [5]:
import base64
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

try:
    from dotenv import load_dotenv
    load_dotenv(Path.cwd().parent / ".env")
    load_dotenv(Path.cwd() / ".env")
except ImportError:
    pass

TEST_DIR = Path("test_items_set")
PREDICTIONS_CSV = TEST_DIR / "test_items_predictions.csv"
CATEGORIES_CSV = TEST_DIR / "categories.csv"

ENRICHED_CSV = TEST_DIR / "test_items_enriched.csv"
ENRICHED_JSONL = TEST_DIR / "test_items_enriched.jsonl"

# Vision-capable model (override if you prefer)
OPENAI_MODEL = os.environ.get("OPENAI_ENRICH_MODEL", "gpt-4o-mini")

ROOT_IDS = {
    "00000000-0000-0000-0000-000000000001",
    "00000000-0000-0000-0000-000000000002",
    "00000000-0000-0000-0000-000000000003",
}
ROOT_TO_DEPT = {
    "00000000-0000-0000-0000-000000000001": "Apparel",
    "00000000-0000-0000-0000-000000000002": "Shoes",
    "00000000-0000-0000-0000-000000000003": "Accessories",
}


def load_taxonomy_lines_by_department() -> dict[str, list[str]]:
    """2-tier lines: 'ParentName: child1, child2, ...' scoped by coarse department."""
    df = pd.read_csv(CATEGORIES_CSV, dtype=str, keep_default_na=False)
    df["parent_id"] = df["parent_id"].replace({"NULL": pd.NA, "": pd.NA})
    tier2 = df[df["parent_id"].isin(ROOT_IDS)].copy()
    tier2["_sort"] = pd.to_numeric(tier2["sort_order"], errors="coerce")
    tier2["dept"] = tier2["parent_id"].map(ROOT_TO_DEPT)

    out: dict[str, list[str]] = {d: [] for d in ROOT_TO_DEPT.values()}
    for dept in ROOT_TO_DEPT.values():
        sub = tier2[tier2["dept"] == dept].sort_values(["_sort", "name"], na_position="last")
        lines: list[str] = []
        for _, parent in sub.iterrows():
            pid = parent["id"]
            kids = df[df["parent_id"] == pid].copy()
            kids["_sort"] = pd.to_numeric(kids["sort_order"], errors="coerce")
            kids = kids.sort_values(["_sort", "name"], na_position="last")
            children = ", ".join(kids["name"].tolist())
            lines.append(f"- {parent['name']}: {children}")
        out[dept] = lines
    return out


TAXONOMY_BY_DEPT = load_taxonomy_lines_by_department()


def taxonomy_block(dept: str) -> str:
    dept = dept.strip()
    if dept not in TAXONOMY_BY_DEPT:
        raise ValueError(f"Unknown department {dept!r}; expected one of {set(TAXONOMY_BY_DEPT)}")
    return "\n".join(TAXONOMY_BY_DEPT[dept])


METADATA_TAGGING_PROMPT_TEMPLATE = '''You are a fashion metadata tagging assistant for a clothing discovery platform.
Infer discovery-oriented semantic tags from the item (image when provided, else text).
For each tag field (themes, occasions, vibes, styles, aesthetic, fit, season, function), include
at least 1 and at most 3 values from that field allowed list below. Never leave any of these arrays empty.
Return ONLY values from the allowed lists. Do not invent values. Prefer specific tags; use "other" only
when nothing else applies.

Also provide "itemName": a single concise catalog title in Title Case (about 3–12 words), clear and
shopper-friendly. Use standard capitalization (not ALL CAPS), no trailing period, no quotation marks,
max 500 characters. When the user message includes a "Name" line to copy exactly into itemName, follow
that instruction instead of inventing a new title.

Also write a short "description": 2–4 plain-text sentences for product detail pages—concrete,
shopper-friendly, neutral tone (no markdown, no bullet lists, do not repeat the tag names as a list).

SUBCATEGORIES (required; use ONLY the FitFolio taxonomy block for this item's coarse category):
Each taxonomy line has the form: "- ParentName: child A, child B, child C, ...".
- "subCategory_1": Must be exactly one **ParentName** from a line in the block (the text before ": ").
  Examples: "Tops", "Casual", "Bags".
- "subCategory_2": Must be exactly one **child** listed after the colon on the **same line** as that parent.
  **Critical:** subCategory_2 MUST belong to the line you chose for subCategory_1—do not mix children from
  different parent lines. The pair (subCategory_1, subCategory_2) must be a valid parent→child pair from
  the taxonomy.
- Copy spelling and punctuation exactly (including "&", "/", and parentheses).
- Choose the pair that best matches the **image** and stays **consistent** with coarse category
  (Apparel / Shoes / Accessories), gender, and color when those help disambiguate.

ALLOWED VALUES - each tag field must have 1-3 values from its list (never empty). Return ONLY values from these lists:
themes: love, athletic, minimalist, classic, luxury, street, utility, rebellious, punk, grunge, gothic, western, bohemian, artistic, nature, nautical, military, outdoor, holiday, christmas, halloween, valentine, academic, youthful, futuristic, graphic, statement, other
occasions: everyday, errands, weekend, home, lounge, sleep, work, office, business_casual, interview, school, study_session, graduation, formal_event, semi_formal, cocktail_event, wedding_guest, date_night, valentines_day, dinner, brunch, party, birthday, night_out, festival, concert, vacation, travel, airport, beach, pool, resort, outdoor, hiking, camping, gym, workout, running, yoga, dance, holiday, christmas, new_year, halloween, family_gathering, picnic, garden_party, other
vibes: cute, romantic, cozy, soft, comfortable, relaxed, laid_back, chill, playful, fun, happy, bright, fresh, clean, calm, dreamy, sweet, delicate, elegant, classy, polished, refined, luxurious, bold, confident, powerful, edgy, cool, sharp, sleek, mysterious, dark, moody, dramatic, rebellious, sensual, flirty, energetic, youthful, nostalgic, timeless, festive, glam, other
styles: casual, basic, minimalist, classic, modern, smart_casual, business_casual, formal, tailored, elegant, chic, luxury, preppy, sporty, athleisure, streetwear, utility, workwear, outdoor, techwear, bohemian, romantic, feminine, masculine, edgy, punk, grunge, gothic, western, vintage, retro, artsy, resort, beachwear, loungewear, sleepwear, graphic, statement, other
aesthetic: old_money, quiet_luxury, clean_girl, minimalist, normcore, coastal, boho, cottagecore, coquette, soft_girl, dark_academia, light_academia, indie, vintage, retro, y2k, grunge, punk, goth, streetwear, skater, tenniscore, balletcore, gorpcore, western, techwear, avant_garde, futuristic, cyber, fairycore, dark_feminine, mob_wife, glam, other
fit: oversized, relaxed_fit, regular_fit, slim_fit, fitted, tailored_fit, boxy, cropped, longline, loose, bodycon, straight, wide_leg, flared, bootcut, tapered, high_waisted, mid_rise, low_rise, a_line, flowy, structured, draped, drop_shoulder, mini, midi, maxi, other
season: spring, summer, fall, winter, all_season, transitional, other
function: layering, outer_layer, base_layer, warmth, breathable, lightweight, weather_protection, performance, stretch, comfort, travel_friendly, packable, easy_care, quick_dry, moisture_wicking, coverage, shape_enhancing, other

FITFOLIO SUBCATEGORY TAXONOMY (coarse category = {dept}) — use ONLY this section for subCategory_1 and subCategory_2:
{taxonomy_block}

OUTPUT: Return ONE JSON object only (no markdown fences) with keys:
itemName (string), description (string),
themes, occasions, vibes, styles, aesthetic, fit, season, function (each an array of 1-3 strings),
subCategory_1 (string), subCategory_2 (string).
'''


def build_enrichment_system_prompt(dept: str) -> str:
    """Full system prompt for a coarse category (Apparel / Shoes / Accessories)."""
    d = dept.strip()
    return METADATA_TAGGING_PROMPT_TEMPLATE.format(
        dept=d,
        taxonomy_block=taxonomy_block(d),
    )



In [7]:
# Preview the exact system prompt sent to the model (after running the setup cell above).
PREVIEW_DEPT = "Apparel"  # try: "Shoes", "Accessories"

_prompt = build_enrichment_system_prompt(PREVIEW_DEPT)
print(f"--- System prompt (coarse category = {PREVIEW_DEPT}) — {len(_prompt)} chars ---\n")
print(_prompt)


--- System prompt (coarse category = Apparel) — 5515 chars ---

You are a fashion metadata tagging assistant for a clothing discovery platform.
Infer discovery-oriented semantic tags from the item (image when provided, else text).
For each tag field (themes, occasions, vibes, styles, aesthetic, fit, season, function), include
at least 1 and at most 3 values from that field allowed list below. Never leave any of these arrays empty.
Return ONLY values from the allowed lists. Do not invent values. Prefer specific tags; use "other" only
when nothing else applies.

Also provide "itemName": a single concise catalog title in Title Case (about 3–12 words), clear and
shopper-friendly. Use standard capitalization (not ALL CAPS), no trailing period, no quotation marks,
max 500 characters. When the user message includes a "Name" line to copy exactly into itemName, follow
that instruction instead of inventing a new title.

Also write a short "description": 2–4 plain-text sentences for product detai

In [4]:
def encode_image_b64(image_path: Path) -> tuple[str, str]:
    """Return (media_type, base64_str) for common image types."""
    suffix = image_path.suffix.lower()
    media = {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".webp": "image/webp",
    }.get(suffix, "image/jpeg")
    data = image_path.read_bytes()
    return media, base64.b64encode(data).decode("ascii")


def build_user_text(row: pd.Series) -> str:
    return (
        "Known signals from our classifier (may be imperfect; trust the image if they conflict):\n"
        f"- id: {row['id']}\n"
        f"- category (coarse): {row['category']}\n"
        f"- gender: {row['gender']}\n"
        f"- color: {row['color']}\n\n"
        "Respond with the JSON object described in the system message."
    )


def enrich_row(client: OpenAI, row: pd.Series, image_path: Path, *, model: str) -> dict:
    dept = row["category"].strip()
    system_prompt = build_enrichment_system_prompt(dept)
    media_type, b64 = encode_image_b64(image_path)
    user_text = build_user_text(row)

    resp = client.chat.completions.create(
        model=model,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": user_text},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{media_type};base64,{b64}",
                        },
                    },
                ],
            },
        ],
    )
    raw = resp.choices[0].message.content
    return json.loads(raw)


def flatten_tags(d: dict) -> dict:
    """Join tag arrays for CSV friendliness."""
    out = dict(d)
    for k in ("themes", "occasions", "vibes", "styles", "aesthetic", "fit", "season", "function"):
        v = out.get(k)
        if isinstance(v, list):
            out[k] = ";".join(str(x) for x in v)
    return out


# --- Run batch ---
# Pause after each item (success or final failure). Override with OPENAI_ENRICH_SLEEP.
POST_ENRICH_SLEEP = float(os.environ.get("OPENAI_ENRICH_SLEEP", "2"))
# Retries per item (rate limits, transient errors). Override with OPENAI_ENRICH_MAX_RETRIES.
MAX_ENRICH_RETRIES = max(1, int(os.environ.get("OPENAI_ENRICH_MAX_RETRIES", "4")))

client = OpenAI()

pred = pd.read_csv(PREDICTIONS_CSV, dtype=str)
records: list[dict] = []
errors: list[tuple[str, str]] = []

for _, row in pred.iterrows():
    img = TEST_DIR / f"{row['id']}.jpeg"
    if not img.is_file():
        img = TEST_DIR / f"{row['id']}.jpg"
    if not img.is_file():
        errors.append((row["id"], "image not found"))
        time.sleep(POST_ENRICH_SLEEP)
        continue

    payload = None
    last_err: Exception | None = None
    for attempt in range(MAX_ENRICH_RETRIES):
        try:
            payload = enrich_row(client, row, img, model=OPENAI_MODEL)
            break
        except Exception as e:
            last_err = e
            if attempt < MAX_ENRICH_RETRIES - 1:
                backoff = min(2**attempt, 30.0)
                print(f"  {row['id']} retry {attempt + 2}/{MAX_ENRICH_RETRIES} after {backoff:.0f}s — {e}")
                time.sleep(backoff)

    if payload is not None:
        rec = {
            "id": row["id"],
            "category": row["category"],
            "gender": row["gender"],
            "color": row["color"],
            **flatten_tags(payload),
        }
        records.append(rec)
        print(row["id"], "→", payload.get("itemName", "")[:60])
    else:
        errors.append((row["id"], str(last_err)))

    time.sleep(POST_ENRICH_SLEEP)

if errors:
    print("Errors:", errors)

out_df = pd.DataFrame.from_records(records)
out_df.to_csv(ENRICHED_CSV, index=False)
print("Wrote", ENRICHED_CSV.resolve())

with ENRICHED_JSONL.open("w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("Wrote", ENRICHED_JSONL.resolve())

out_df


item_1 → Salomon Trail Running Shoes
item_2 → Men's Blue Zip-Up Hoodie
item_3 → Men's Beige Quarter-Zip Sweater
item_4 → Graphic Print White T-Shirt
item_5 → Athletic Crop Top and Shorts Set
item_6 → Soft Zip-Up Cropped Sweatshirt
item_7 → Elegant White Mini Dress
item_8 → Men's Blue Denim Jacket and Pants Set
item_9 → Men's Black Puffer Jacket
item_10 → Brown Pointed Toe Heels
item_11 → Black Crew Socks Set
item_12 → Classic Black Cap
item_13 → Men's Light Blue Button-Up Shirt
item_14 → Men's Grey Chinos
item_15 → Men's Green Shorts
item_16 → Men's Black Leather Loafers
item_17 → Light Pink Long Sleeve Shirt
item_18 → Men's Blue Hoodie with Logo
item_19 → Citizen Eco-Drive Chronograph Watch
item_20 → Classic Men's Polo Shirt
Wrote /Users/akudenko/Documents/FitFolio/fitfolio/fitfolio-ml-api/notebooks/test_items_set/test_items_enriched.csv
Wrote /Users/akudenko/Documents/FitFolio/fitfolio/fitfolio-ml-api/notebooks/test_items_set/test_items_enriched.jsonl


,id,category,gender,color,itemName,description,themes,occasions,vibes,styles,aesthetic,fit,season,function,subCategory_1,subCategory_2
0,item_1,Shoes,men,white,Salomon Trail Running Shoes,Experience comfort and performance with these ...,athletic;utility;outdoor,gym;workout;running,comfortable;relaxed;energetic,sporty;athleisure;utility,gorpcore;futuristic;techwear,regular_fit;tailored_fit;loose,spring;summer;fall,breathable;lightweight;weather_protection,Athletic,Running Shoes
1,item_2,Apparel,men,blue,Men's Blue Zip-Up Hoodie,Stay comfortable and stylish with this men's b...,athletic,everyday;gym;workout,comfortable;relaxed;cool,casual;sporty;streetwear,activewear;modern;minimalist,regular_fit;boxy;relaxed_fit,all_season;transitional,outer_layer;comfort;performance,Outerwear,Hoodies
2,item_3,Apparel,men,beige,Men's Beige Quarter-Zip Sweater,This men's quarter-zip sweater in beige offers...,classic;minimalist;utility,everyday;work;weekend,comfortable;relaxed;elegant,casual;smart_casual;classic,clean_girl;minimalist;retro,regular_fit;fitted;tailored_fit,fall;winter;all_season,layering;comfort;warmth,Tops,Sweaters
3,item_4,Apparel,men,white,Graphic Print White T-Shirt,This stylish white t-shirt features a playful ...,graphic;casual;youthful,everyday;weekend;casual,fun;playful;bright,casual;basic;streetwear,clean_girl;streetwear;modern,regular_fit;fitted;short,spring;summer;all_season,comfort;lightweight;breathable,Tops,T-Shirts
4,item_5,Apparel,women,red,Athletic Crop Top and Shorts Set,Elevate your workout wardrobe with this stylis...,athletic,gym;workout;running,comfortable;energetic;cool,sporty;athleisure;casual,streetwear;modern;y2k,regular_fit;relaxed_fit;fitted,summer;all_season,performance;breathable;comfort,Activewear,Workout Tops
5,item_6,Apparel,women,beige,Soft Zip-Up Cropped Sweatshirt,This cropped zip-up sweatshirt combines comfor...,casual;athletic;minimalist,everyday;weekend;lounge,comfortable;relaxed;laid-back,casual;athleisure;streetwear,minimalist;coastal;soft_girl,cropped;relaxed_fit;boxy,spring;fall;all_season,comfort;layering;lightweight,Tops,Sweaters
6,item_7,Apparel,women,white,Elegant White Mini Dress,This elegant white mini dress features a fitte...,classic;elegant,cocktail_event;party;wedding_guest,classy;timeless;elegant,formal;feminine;chic,clean_girl;soft_girl;elegant,fitted;mini;regular_fit,summer;spring;all_season,comfort;lightweight;layering,Dresses & Jumpsuits,Dresses
7,item_8,Apparel,men,blue,Men's Blue Denim Jacket and Pants Set,This trendy men's denim jacket features a rela...,classic;street;utility,everyday;weekend;casual,cool;relaxed;edgy,casual;streetwear;vintage,retro;grunge;indie,oversized;relaxed_fit;wide_leg,spring;fall;all_season,layering;comfort;travel_friendly,Tops,Jackets
8,item_9,Apparel,men,black,Men's Black Puffer Jacket,Stay warm and stylish with this men's black pu...,utility;athletic;minimalist,everyday;weekend;outdoor,comfortable;relaxed;cool,casual;utility;activewear,minimalist;streetwear;other,regular_fit;relaxed_fit;other,fall;winter;all_season,warmth;weather_protection;lightweight,Outerwear,Jackets
9,item_10,Accessories,women,brown,Brown Pointed Toe Heels,These elegant brown pointed toe heels are perf...,luxury;classic;statement,formal_event;party;night_out,elegant;classy;confident,formal;chic;tailored,clean_girl;vintage;coquette,fitted;slim_fit;structured,all_season;fall;winter,comfort;lift;elegance,Footwear,Heels


### Testing

- **`test_openai_gpt4o`** — one image / one call (smoke test).
- **`run_gpt_enrichment_test_set`** — all rows in `test_items_predictions.csv` (~20 items); writes **`test_items_set/model_outputs/openai_<model>.csv`**, **`.json`** (array), and **`.jsonl`**.

Run the batch cell above first so `enrich_row` and `flatten_tags` exist.

In [13]:
def test_openai_gpt4o(image_path=None, row=None, model="gpt-4o"):
    """One enrichment call with OpenAI GPT-4o (vision + JSON).

    Prerequisites: run the **setup** cell and the **enrich-code-run** cell (`enrich_row`, `flatten_tags`, …).
    """
    if row is None:
        row = pd.Series(
            {
                "id": "test_item",
                "category": "Apparel",
                "gender": "men",
                "color": "blue",
            }
        )
    if image_path is None:
        image_path = TEST_DIR / "item_1.jpeg"
    image_path = Path(image_path)
    client = OpenAI()
    return enrich_row(client, row, image_path, model=model)


def run_gpt_enrichment_test_set(
    model="gpt-4o",
    *,
    predictions_csv=None,
    out_dir=None,
    out_slug=None,
    post_sleep=None,
    max_retries=None,
):
    """Enrich **all** rows in the test predictions CSV with the given OpenAI model.

    Saves under ``test_items_set/model_outputs/``:

    - ``{out_slug}.csv`` — tags joined with ``;``
    - ``{out_slug}.json`` — one JSON **array** of all records
    - ``{out_slug}.jsonl`` — one JSON object per line
    """
    predictions_csv = Path(predictions_csv or PREDICTIONS_CSV)
    if out_slug is None:
        out_slug = "openai_" + model.replace("/", "-").replace(" ", "_")
    if out_dir is None:
        out_dir = TEST_DIR / "model_outputs"
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    csv_path = out_dir / f"{out_slug}.csv"
    json_path = out_dir / f"{out_slug}.json"
    jsonl_path = out_dir / f"{out_slug}.jsonl"

    if post_sleep is None:
        post_sleep = float(os.environ.get("OPENAI_ENRICH_SLEEP", "2"))
    if max_retries is None:
        max_retries = max(1, int(os.environ.get("OPENAI_ENRICH_MAX_RETRIES", "4")))

    client = OpenAI()
    pred = pd.read_csv(predictions_csv, dtype=str)
    records = []
    errors = []

    for _, row in pred.iterrows():
        img = TEST_DIR / f"{row['id']}.jpeg"
        if not img.is_file():
            img = TEST_DIR / f"{row['id']}.jpg"
        if not img.is_file():
            errors.append((row["id"], "image not found"))
            time.sleep(post_sleep)
            continue

        payload = None
        last_err = None
        for attempt in range(max_retries):
            try:
                payload = enrich_row(client, row, img, model=model)
                break
            except Exception as e:
                last_err = e
                if attempt < max_retries - 1:
                    backoff = min(2**attempt, 30.0)
                    print(f"  {row['id']} retry {attempt + 2}/{max_retries} after {backoff:.0f}s — {e}")
                    time.sleep(backoff)

        if payload is not None:
            rec = {
                "provider": "openai",
                "model": model,
                "id": row["id"],
                "category": row["category"],
                "gender": row["gender"],
                "color": row["color"],
                **flatten_tags(payload),
            }
            records.append(rec)
            print(row["id"], "→", payload.get("itemName", "")[:60])
        else:
            errors.append((row["id"], str(last_err)))

        time.sleep(post_sleep)

    if errors:
        print("Errors:", errors)

    out_df = pd.DataFrame.from_records(records)
    out_df.to_csv(csv_path, index=False)
    print("Wrote", csv_path.resolve())

    with json_path.open("w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)
    print("Wrote", json_path.resolve())

    with jsonl_path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("Wrote", jsonl_path.resolve())

    return out_df


# Examples (after setup + enrich-code-run):
# one = test_openai_gpt4o()
# print(json.dumps(one, indent=2, ensure_ascii=False))

df_all = run_gpt_enrichment_test_set("gpt-4o")
df_all


item_1 → Men's Performance Running Shoes
item_2 → Men's Navy Athletic Tracksuit
item_3 → Men's Beige Quarter-Zip Sweater
item_4 → White Graphic T-Shirt with Peach Design
item_5 → Red Women's Sports Bra
item_6 → Women's Beige Zip-Up Sweatshirt
item_7 → Women's White Sleeveless Mini Dress
item_8 → Blue Denim Jacket and Jeans Set
item_9 → Men's Black Puffer Jacket
item_10 → Brown Pointed Toe Heeled Pumps
item_11 → Men's Blue Ombre Swim Shorts
item_12 → Black Logo Baseball Cap
item_13 → Men's Light Blue Button-Up Shirt
item_14 → Men's Grey Chino Pants
item_15 → Men's Green Drawstring Shorts
item_16 → Men's Black Leather Loafers
item_17 → Men's Pink Button-Up Shirt
item_18 → Men's Blue Hoodie with Logo
item_19 → Men's Eco-Drive Chronograph Watch
item_20 → Men's Blue Polo Shirt
Wrote /Users/akudenko/Documents/FitFolio/fitfolio/fitfolio-ml-api/notebooks/test_items_set/model_outputs/openai_gpt-4o.csv
Wrote /Users/akudenko/Documents/FitFolio/fitfolio/fitfolio-ml-api/notebooks/test_items_set/mod

,provider,model,id,category,gender,color,itemName,description,themes,occasions,vibes,styles,aesthetic,fit,season,function,subCategory_1,subCategory_2
0,openai,gpt-4o,item_1,Shoes,men,white,Men's Performance Running Shoes,These men's performance running shoes are desi...,athletic;utility,gym;running;outdoor,energetic;powerful,sporty;athleisure,techwear;performance,regular_fit,all_season,performance;breathable;lightweight,Athletic,Running Shoes
1,openai,gpt-4o,item_2,Apparel,men,blue,Men's Navy Athletic Tracksuit,This men's navy tracksuit features a hoodie wi...,athletic;street,gym;casual;everyday,cool;comfortable,sporty;casual,streetwear;sporty,regular_fit;comfortable,all_season,comfort;performance,Activewear,Tracksuits
2,openai,gpt-4o,item_3,Apparel,men,beige,Men's Beige Quarter-Zip Sweater,This men's beige sweater features a classic qu...,classic;minimalist,everyday;work;weekend,comfortable;relaxed;clean,casual;smart_casual;classic,quiet_luxury;minimalist,regular_fit,fall;winter;transitional,layering;comfort,Tops,Sweaters
3,openai,gpt-4o,item_4,Apparel,men,white,White Graphic T-Shirt with Peach Design,This white T-shirt features a playful peach gr...,graphic;youthful,everyday;casual;school,cute;fun;fresh,casual;basic;streetwear,minimalist;streetwear,regular_fit,summer;spring,breathable;comfort,Tops,T-Shirts
4,openai,gpt-4o,item_5,Apparel,women,red,Red Women's Sports Bra,This red women's sports bra is designed for ac...,athletic;minimalist,gym;workout;running,energetic;comfortable,sporty;athleisure,minimalist;streetwear,fitted,all_season,performance;comfort;breathable,Activewear,Sports Bras
5,openai,gpt-4o,item_6,Apparel,women,beige,Women's Beige Zip-Up Sweatshirt,This beige zip-up sweatshirt offers a relaxed ...,minimalist;street,everyday;weekend;lounge,comfortable;relaxed;chill,casual;streetwear,minimalist;normcore,relaxed_fit;cropped,spring;fall;transitional,comfort;layering,Tops,Sweatshirts
6,openai,gpt-4o,item_7,Apparel,women,white,Women's White Sleeveless Mini Dress,This elegant white mini dress features a sleev...,classic;minimalist,semi_formal;night_out;dinner,elegant;classy;polished,chic;classic;elegant,quiet_luxury;minimalist,fitted;mini,summer;spring,comfort;layering,Dresses & Jumpsuits,Dresses
7,openai,gpt-4o,item_8,Apparel,men,blue,Blue Denim Jacket and Jeans Set,This blue denim set features a classic jacket ...,classic;casual,everyday;weekend;vacation,cool;timeless;relaxed,casual;streetwear;denim,vintage;streetwear;normcore,relaxed_fit;wide_leg,fall;spring;transitional,layering;comfort,Outerwear,Jackets
8,openai,gpt-4o,item_9,Apparel,men,black,Men's Black Puffer Jacket,This men's black puffer jacket is designed for...,utility;minimalist,everyday;outdoor;travel,cool;sleek;comfortable,casual;utility;outdoor,minimalist;techwear,regular_fit,winter;fall,warmth;outer_layer,Outerwear,Jackets
9,openai,gpt-4o,item_10,Accessories,women,brown,Brown Pointed Toe Heeled Pumps,These brown heeled pumps feature a classic poi...,classic;luxury,formal_event;office;night_out,elegant;classy;polished,classic;formal;chic,quiet_luxury;timeless,fitted,all_season,comfort,Shoes,Heels


In [18]:
import json
import os
import time
from pathlib import Path

import pandas as pd
from PIL import Image

try:
    from dotenv import load_dotenv
    load_dotenv(Path.cwd().parent / ".env")
    load_dotenv(Path.cwd() / ".env")
except ImportError:
    pass

from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

GEMINI_MODEL = os.environ.get("GEMINI_ENRICH_MODEL", "gemini-1.5-pro")


def enrich_row_gemini(client, row: pd.Series, image_path: Path, *, model: str) -> dict:
    dept = row["category"].strip()
    system_prompt = build_enrichment_system_prompt(dept)
    user_text = build_user_text(row)

    # Gemini expects structured "parts"
    image_bytes = image_path.read_bytes()

    response = client.models.generate_content(
        model=model,
        contents=[
            {
                "role": "user",
                "parts": [
                    {"text": system_prompt},
                    {"text": user_text},
                    {
                        "inline_data": {
                            "mime_type": "image/jpeg",
                            "data": image_bytes,
                        }
                    },
                ],
            }
        ],
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",  # IMPORTANT
        ),
    )

    raw = response.text.strip()

    # Safety cleanup
    raw = raw.replace("```json", "").replace("```", "").strip()

    return json.loads(raw)


def test_gemini(image_path=None, row=None, model=GEMINI_MODEL):
    """One enrichment call with Gemini (vision + JSON)."""
    if row is None:
        row = pd.Series(
            {
                "id": "test_item",
                "category": "Apparel",
                "gender": "men",
                "color": "blue",
            }
        )
    if image_path is None:
        image_path = TEST_DIR / "item_1.jpeg"

    image_path = Path(image_path)
    model_client = genai.GenerativeModel(model)
    return enrich_row_gemini(model_client, row, image_path, model=model)


def run_gemini_enrichment_test_set(
    model=GEMINI_MODEL,
    *,
    predictions_csv=None,
    out_dir=None,
    out_slug=None,
    post_sleep=None,
    max_retries=None,
):
    """
    Enrich all rows in the test predictions CSV with Gemini.

    Saves under test_items_set/model_outputs/:
    - {out_slug}.csv
    - {out_slug}.json
    - {out_slug}.jsonl
    """
    predictions_csv = Path(predictions_csv or PREDICTIONS_CSV)

    if out_slug is None:
        out_slug = "gemini_" + model.replace("/", "-").replace(" ", "_")

    if out_dir is None:
        out_dir = TEST_DIR / "model_outputs"

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    csv_path = out_dir / f"{out_slug}.csv"
    json_path = out_dir / f"{out_slug}.json"
    jsonl_path = out_dir / f"{out_slug}.jsonl"

    if post_sleep is None:
        post_sleep = float(os.environ.get("GEMINI_ENRICH_SLEEP", "2"))

    if max_retries is None:
        max_retries = max(1, int(os.environ.get("GEMINI_ENRICH_MAX_RETRIES", "4")))

    pred = pd.read_csv(predictions_csv, dtype=str)
    model_client = genai.GenerativeModel(model)

    records: list[dict] = []
    errors: list[tuple[str, str]] = []

    for _, row in pred.iterrows():
        img = TEST_DIR / f"{row['id']}.jpeg"
        if not img.is_file():
            img = TEST_DIR / f"{row['id']}.jpg"
        if not img.is_file():
            img = TEST_DIR / f"{row['id']}.png"

        if not img.is_file():
            errors.append((row["id"], "image not found"))
            time.sleep(post_sleep)
            continue

        payload = None
        last_err: Exception | None = None

        for attempt in range(max_retries):
            try:
                payload = enrich_row_gemini(model_client, row, img, model=model)
                break
            except Exception as e:
                last_err = e
                if attempt < max_retries - 1:
                    backoff = min(2**attempt, 30.0)
                    print(f"  {row['id']} retry {attempt + 2}/{max_retries} after {backoff:.0f}s — {e}")
                    time.sleep(backoff)

        if payload is not None:
            rec = {
                "provider": "gemini",
                "model": model,
                "id": row["id"],
                "category": row["category"],
                "gender": row["gender"],
                "color": row["color"],
                **flatten_tags(payload),
            }
            records.append(rec)
            print(row["id"], "→", payload.get("itemName", "")[:60])
        else:
            errors.append((row["id"], str(last_err)))

        time.sleep(post_sleep)

    if errors:
        print("Errors:", errors)

    out_df = pd.DataFrame.from_records(records)
    out_df.to_csv(csv_path, index=False)
    print("Wrote", csv_path.resolve())

    with json_path.open("w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)
    print("Wrote", json_path.resolve())

    with jsonl_path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("Wrote", jsonl_path.resolve())

    return out_df

AttributeError: module 'google.genai' has no attribute 'configure'

In [ ]:


from openai import OpenAI
from google import genai
from google.genai import types

openai_client = OpenAI()
gemini_client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

OPENAI_MODEL = os.environ.get("OPENAI_ENRICH_MODEL", "gpt-4o")
OPENAI_MODEL_SIMPLE = os.environ.get("OPENAI_ENRICH_MODEL", "gpt-4o-mini")
OPENAI_MODEL_COMPLEX = os.environ.get("OPENAI_ENRICH_MODEL", "gpt-4.1")
GEMINI_MODEL = os.environ.get("GEMINI_ENRICH_MODEL", "gemini-2.5-flash")

In [ ]:
def get_mime_type(path: Path) -> str:
    return {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".webp": "image/webp",
    }.get(path.suffix.lower(), "image/jpeg")


def enrich_row_openai(client, row: pd.Series, image_path: Path, *, model: str) -> dict:
    dept = row["category"].strip()
    system_prompt = build_enrichment_system_prompt(dept)
    media_type, b64 = encode_image_b64(image_path)
    user_text = build_user_text(row)

    resp = client.chat.completions.create(
        model=model,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": user_text},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{media_type};base64,{b64}",
                        },
                    },
                ],
            },
        ],
    )

    raw = resp.choices[0].message.content
    return json.loads(raw)


def enrich_row_gemini(client, row: pd.Series, image_path: Path, *, model: str) -> dict:
    dept = row["category"].strip()
    system_prompt = build_enrichment_system_prompt(dept)
    user_text = build_user_text(row)

    image_bytes = image_path.read_bytes()
    mime_type = get_mime_type(image_path)

    response = client.models.generate_content(
        model=model,
        contents=[
            {
                "role": "user",
                "parts": [
                    {"text": system_prompt},
                    {"text": user_text},
                    {
                        "inline_data": {
                            "mime_type": mime_type,
                            "data": image_bytes,
                        }
                    },
                ],
            }
        ],
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
        ),
    )

    raw = response.text.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()

    return json.loads(raw)

In [ ]:

def run_model_enrichment(
    provider: str,
    model: str,
    *,
    predictions_csv=None,
    out_dir=None,
    out_slug=None,
    post_sleep=2,
    max_retries=4,
):
    predictions_csv = Path(predictions_csv or PREDICTIONS_CSV)

    if out_slug is None:
        out_slug = f"{provider}_{model.replace('/', '-')}".replace(" ", "_")

    if out_dir is None:
        out_dir = TEST_DIR / "model_outputs"

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    csv_path = out_dir / f"{out_slug}.csv"
    json_path = out_dir / f"{out_slug}.json"
    jsonl_path = out_dir / f"{out_slug}.jsonl"

    pred = pd.read_csv(predictions_csv, dtype=str)

    records = []
    errors = []

    # Select provider function
    if provider == "openai":
        client = openai_client
        enrich_fn = enrich_row_openai
    elif provider == "gemini":
        client = gemini_client
        enrich_fn = enrich_row_gemini
    else:
        raise ValueError(f"Unknown provider: {provider}")

    for _, row in pred.iterrows():
        img = TEST_DIR / f"{row['id']}.jpeg"
        if not img.is_file():
            img = TEST_DIR / f"{row['id']}.jpg"
        if not img.is_file():
            img = TEST_DIR / f"{row['id']}.png"

        if not img.is_file():
            errors.append((row["id"], "image not found"))
            time.sleep(post_sleep)
            continue

        payload = None
        last_err = None

        for attempt in range(max_retries):
            try:
                payload = enrich_fn(client, row, img, model=model)
                break
            except Exception as e:
                last_err = e
                if attempt < max_retries - 1:
                    backoff = min(2**attempt, 30)
                    print(f"{row['id']} retry {attempt+2}/{max_retries} after {backoff}s — {e}")
                    time.sleep(backoff)

        if payload:
            rec = {
                "provider": provider,
                "model": model,
                "id": row["id"],
                "category": row["category"],
                "gender": row["gender"],
                "color": row["color"],
                **flatten_tags(payload),
            }
            records.append(rec)
            print(row["id"], "→", payload.get("itemName", "")[:60])
        else:
            errors.append((row["id"], str(last_err)))

        time.sleep(post_sleep)

    if errors:
        print("Errors:", errors)

    df = pd.DataFrame.from_records(records)

    df.to_csv(csv_path, index=False)
    with json_path.open("w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)
    with jsonl_path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print("Saved:", csv_path)

    return df

In [22]:
# =========================
# RUN ALL MODELS
# =========================

def run_all_models():
    results = {}

    print("\n=== OpenAI ===")
    results["openai"] = run_model_enrichment(
        provider="openai",
        model=OPENAI_MODEL,
    )

    print("\n=== Gemini ===")
    results["gemini"] = run_model_enrichment(
        provider="gemini",
        model=GEMINI_MODEL,
    )

    return results


results = run_all_models()


=== OpenAI ===
item_1 → Men's Trail Running Shoes
item_2 → Men's Blue Hoodie Tracksuit
item_3 → Men's Beige Quarter-Zip Sweater
item_4 → White Graphic T-Shirt
item_5 → Red Sports Bra
item_6 → Women's Beige Cropped Half-Zip Sweatshirt
item_7 → White Sleeveless Mini Dress
item_8 → Men's Blue Denim Jacket
item_9 → Men's Black Puffer Jacket
item_10 → Brown Pointed-Toe High Heels
item_11 → Men's Ombre Swim Trunks
item_12 → Black Logo Embroidered Baseball Cap
item_13 → Men's Light Blue Button-Up Shirt
item_14 → Men's Grey Chino Pants
item_15 → Men's Green Casual Shorts
item_16 → Men's Black Leather Loafers
item_17 → Men's Pink Long Sleeve Button-Up Shirt
item_18 → Blue Team Logo Hoodie
item_19 → Men's Blue Chronograph Watch
item_20 → Men's Blue Polo Shirt
Saved: test_items_set/model_outputs/openai_gpt-4o.csv

=== Gemini ===
item_1 retry 2/4 after 1s — 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-pro is not found for API version v1beta, or is not supported for generat

In [26]:
def run_gemini():
    results = {}

    print("\n=== Gemini ===")
    results["gemini"] = run_model_enrichment(
        provider="gemini",
        model=GEMINI_MODEL,
    )

    return results


results = run_gemini()


=== Gemini ===
item_1 → Salomon XT-6 GORE-TEX Trail Running Shoes
item_2 retry 2/4 after 1s — 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
item_2 → Men's Full-Zip Hoodie with Bold Logo
item_3 → Men's Classic Quarter-Zip Pullover Top
item_4 → Women's White Nike Apple Graphic Short Sleeve T-Shirt
item_5 → Women's Red Performance Sports Bra
item_6 → Women's Cropped Quarter-Zip Mock Neck Sweatshirt
item_7 → Elegant Ruched Boat Neck Mini Dress
item_8 → Classic Blue Oversized Denim Jacket
item_9 → Men's Black Hooded Puffer Jacket
item_10 → Elegant Brown Pointed-Toe Stiletto Heels
item_11 retry 2/4 after 1s — 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
item_11 → Men's Ombre Blue Drawstring Swim S

In [30]:
def run_openai():
    results = {}

    print("\n=== OpenAI ===")
    results["openai"] = run_model_enrichment(
        provider="openai",
        model=OPENAI_MODEL_SIMPLE,
    )

    return results


results = run_openai()

# Should be === OPENAI ===


=== Gemini ===
item_1 → Salomon Men's Trail Running Shoes
item_2 → Adidas Men's Zip-Up Hoodie
item_3 → Beige Quarter-Zip Pullover
item_4 → Graphic T-Shirt with Orange Design
item_5 → Women's Athletic Outfit with Sports Bra and Shorts
item_6 → Cropped Pullover Sweatshirt
item_7 → White Sleeveless Mini Dress
item_8 → Blue Denim Jacket and Wide Leg Pants
item_9 → Black Puffer Jacket
item_10 → Brown Pointed Toe High Heels
item_11 → Ombre Swim Trunks
item_12 → Classic Black Baseball Cap
item_13 → Men's Light Blue Button-Up Shirt
item_14 → Slim Fit Grey Trousers
item_15 → Men's Green Lounge Shorts
item_16 → Men's Black Leather Loafers
item_17 → Men's Pink Long Sleeve Shirt
item_18 → Men's Blue Hoodie with Logo
item_19 → Citizen Eco-Drive Radio Controlled Watch
item_20 → Men's Blue Polo Shirt
Saved: test_items_set/model_outputs/openai_gpt-4o-mini.csv


In [31]:
def run_openai():
    results = {}

    print("\n=== OpenAI ===")
    results["openai"] = run_model_enrichment(
        provider="openai",
        model=OPENAI_MODEL_COMPLEX,
    )

    return results


results = run_openai()


=== OpenAI ===
item_1 → Salomon White Trail Running Shoes
item_2 → Men's Blue Zip-Up Athletic Hoodie
item_3 → Men's Beige Quarter-Zip Pullover Sweater
item_4 → White Peach Graphic Short Sleeve T-Shirt
item_5 → Red Racerback Sports Bra for Women
item_6 → Beige Cropped Half-Zip Sweatshirt
item_7 → Sleeveless White Ruched Mini Dress
item_8 → Blue Relaxed Fit Denim Jacket
item_9 → Men's Black Hooded Puffer Jacket
item_10 → Brown Pointed-Toe Stiletto Pumps
item_11 → Noir Black Leather Men's Belt
item_12 → Black Tommy Hilfiger Embroidered Logo Cap
item_13 → Men's Light Blue Button-Up Oxford Shirt
item_14 → Men's Grey Classic Chino Pants
item_15 → Men's Green Drawstring Sweat Shorts
item_16 → Men's Black Leather Loafer Dress Shoes
item_17 → Men's Pink Long Sleeve Button-Up Shirt
item_18 → Men's Blue Toronto Blue Jays Logo Hoodie
item_19 → Citizen Eco-Drive Blue Radio Controlled Chronograph Watch
item_20 → Men's Blue Classic Short Sleeve Polo Shirt
Saved: test_items_set/model_outputs/openai_g